# Project.py

In [1]:
import logging
logging.basicConfig(
    level=logging.INFO,
    force=True,
)

import warnings
warnings.filterwarnings(action='ignore')

from polymerist.rdutils import disable_kekulized_drawing
disable_kekulized_drawing()

INFO:rdkit:Enabling RDKit 2023.09.6 jupyter extensions
INFO:numexpr.utils:Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [2]:
from pathlib import Path
from src.project import (
    PolymerBuildProject,
    mechanism_established,
)

project_path = Path('polyID_test')
# project_path = Path('polyID_production')
project = PolymerBuildProject.get_project(project_path)

[17:32:28] WARNING: not removing hydrogen atom with dummy atom neighbors
INFO:polymerist.smileslib.functgroups:Loading functional group SMARTS data from LUT
INFO:root:Initializing reaction template 1/8 ("polyester")
INFO:root:Initializing test reactants for validation
INFO:root:Initializing reaction template 2/8 ("polyamide")
INFO:root:Initializing test reactants for validation
INFO:root:Initializing reaction template 3/8 ("polyimide")
INFO:root:Initializing test reactants for validation
INFO:root:Initializing reaction template 4/8 ("polycarbonate_phosgene")
INFO:root:Initializing test reactants for validation
INFO:root:Initializing reaction template 5/8 ("polycarbonate_nonphosgene")
INFO:root:Initializing test reactants for validation
INFO:root:Initializing reaction template 6/8 ("polyurethane_isocyanate")
INFO:root:Initializing test reactants for validation
INFO:root:Initializing reaction template 7/8 ("polyurethane_nonisocyanate")
INFO:root:Initializing test reactants for validation

In [ ]:
project.print_status(detailed=True)

#### Run Jobs

In [ ]:
project.run()

In [ ]:
project.run(
    names=[
        # 'polymerize',
        # 'oligomerize',
        # 'pack_lattice',
        # 'to_interchange',
        'md_export'
    ],
    # jobs=[
    #     project.open_job(id='214e8512b6218c21668ce8c19c6bf359')
    # ]
)

## Inspect jobs

### Diagnosing PDB substructure match issues

In [3]:
from src.project import openff_pdb_read_failed

pdb_err_jobs = [job for job in project if openff_pdb_read_failed(job)]
pdb_err_jobs

[Job(project=PolymerBuildProject('/home/timber/Documents/Python/NREL_polymers/polyID_test'), statepoint={'smiles_explicit': '[H]-[C](-[H])=[C](-[H])-[c]1:[c](-[H]):[c](-[H]):[c](-[C](-[H])(-[H])-[C](-[H])(-[H])-[H]):[c](-[H]):[c]:1-[H]', 'DOP': 3, 'n_atoms_max': 20000, 'pcharge_method': 'Espaloma-AM1-BCC', 'forcefield': 'openff_unconstrained-2.0.0.offxml', 'minimize_oligomer': True, 'use_switching_function': False, 'switch_width_nm': 0.1, 'nonbonded_cutoff_nm': 0.9, 'box_padding_nm': 0.0}),
 Job(project=PolymerBuildProject('/home/timber/Documents/Python/NREL_polymers/polyID_test'), statepoint={'smiles_explicit': '[H]-[c]1:[c](-[H]):[c](-[C](-[H])(-[H])-[c]2:[c](-[H]):[c](-[H]):[c](-[N](-[H])-[H]):[c](-[H]):[c]:2-[H]):[c](-[H]):[c](-[H]):[c]:1-[N](-[H])-[H].[H]-[c]1:[c](-[H]):[c](-[C](-[c]2:[c](-[H]):[c](-[H]):[c](-[O]-[C](=[O])-[c]3:[c](-[H]):[c](-[H]):[c]4:[c](:[c]:3-[H])-[C](=[O])-[O]-[C]-4=[O]):[c](-[H]):[c]:2-[H])(-[C](-[F])(-[F])-[F])-[C](-[F])(-[F])-[F]):[c](-[H]):[c](-[H]):[c]:1

### Checking that aromaticity is being respected

In [ ]:
from src.utils.dataIO import read_rxn_mapping_data
from polymerist.rdutils.reactions.reactions import AnnotatedReaction


RXN_DIR : Path = Path('src/reactions')
# RXN_PATHNAME : str = 'rxn_smarts.json'
RXN_PATHNAME : str = 'rxns_polyID.json'

show : bool = not True

# load reactions
rxns : dict[str, AnnotatedReaction] = {}
for rxnname, smarts in read_rxn_mapping_data(RXN_DIR / RXN_PATHNAME).items():
    rxn = AnnotatedReaction.from_smarts(smarts)
    rxns[rxnname] = rxn 
    if show:
        print(rxnname)
        display(rxn)

In [ ]:
from rdkit import Chem
from rdkit.Chem.rdmolops import SANITIZE_ALL, AROMATICITY_MDL
from polymerist.rdutils.sanitization import sanitize_mol

from polymerist.rdutils.reactions.reactors import PolymerizationReactor
from polymerist.rdutils.reactions.fragment import CutMinimumCostBondsStrategy

build_jobs = [job for job in project if mechanism_established(job)]
job = build_jobs[0]
job = project.open_job(id='455fc6cfdb0fc1e7e8ce8c61ce512567')

monomers = PolymerBuildProject.sanitized_mol_from_smiles(job.sp.smiles_explicit, separate_mols=True)
for m in monomers:
    display(m)

In [ ]:
job = project.open_job(id='57825dbdb6cab389c8f39ad2db3b4c1d')

In [ ]:
import pandas as pd

records : list[dict] = []
for job in project:
    op_times = job.doc.get(PolymerBuildProject.OP_TIME_RECORD_NAME)
    if op_times is not None:
        op_times['Job ID'] = job.id
        records.append(op_times)